# This notebook shows how to compute the phonon dispersion and density of states of bulk silicon via DFPT
### Import the necessary libraries

**Cells below are not executed here**: DFPT cost is dominated by the number of atomic-displacement perturbations times q-points, not by anything elkpy controls -- even the smallest meaningful grid (`ngridq=(2,2,2)`, 2 atoms/cell) took ~11-13 minutes per call on the machine `tests/test_calculation_si_phonons.py` was last verified on. Run it yourself with `jupyter nbconvert --to notebook --execute --inplace notebooks/03_phonon_dispersion_and_dos.ipynb`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from elkpy.structure import Structure

SI_AVEC = [(5.13, 5.13, 0.00), (5.13, 0.00, 5.13), (0.00, 5.13, 5.13)]
SI_SPECIES = {"Si": [(0.0, 0.0, 0.0), (0.25, 0.25, 0.25)]}

structure = Structure(SI_AVEC, SI_SPECIES)
calc = structure.get_calculation("_scratch/si", xc="PW", ngridk=(2, 2, 2))

### Dynamical matrix and phonon frequencies (density functional perturbation theory)
$$ D_{\kappa\alpha,\kappa'\beta}(\mathbf q)=\frac1{\sqrt{M_\kappa M_{\kappa'}}}\,\frac{\partial^2 E}{\partial u^*_{\kappa\alpha}(\mathbf q)\,\partial u_{\kappa'\beta}(\mathbf q)},
\qquad \det\big[D(\mathbf q)-\omega_\nu^2(\mathbf q)\,\mathbb 1\big]=0 $$
2 atoms/cell -> 6 phonon branches $\omega_\nu(\mathbf q)$; the 3 acoustic branches go to $\omega\to0$ at $\Gamma$.

In [ ]:
distances, frequencies = calc.get_phonon_dispersion(
    vertices=[(0.0, 0.0, 0.0), (0.5, 0.0, 0.0)], ngridq=(2, 2, 2), npoints=50
)

fig, ax = plt.subplots(figsize=(6, 4))
for branch in frequencies:
    ax.plot(distances, branch)
ax.set_ylabel(r"$\omega(q)$ (Ha)")
plt.show()

### Phonon density of states

In [ ]:
frequencies_dos, phdos = calc.get_phonon_dos(ngridq=(2, 2, 2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(frequencies_dos, phdos)
ax.set_xlabel(r"$\omega$ (Ha)")
ax.set_ylabel("phonon DOS")
plt.show()